In [ ]:
import pandas as pd
from google import genai
import time

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
INPUT_PATH = r"/content/drive/MyDrive/gemini/TEXT/dataset_anonim_phase3.xlsx"
OUTPUT_PATH = r"/content/drive/MyDrive/gemini/TEXT/dataset_anonim_phase4.xlsx"
API_KEY = ""  #Your API Key Gemini

# Prompt dasar / perintah
PERINTAH = (
    "cari mana yang merupakan kalimat tanya dalam kalimat yang akan saya berikan. saya perlu membersihkan kalimat ini dari sapaan, salam pembuka, salam penutup, perkenalan, dan informasi tambahan lainnya. saya perlu mendapatkan kalimat tanya yang sesungguhnya, karena saya ingin mendapatkan apa intent dari keseluruhan kalimat ini. kalimat ini saya peroleh dari seseorang yang melakukan chat ke national statistics office perilhal konsultasi data statistik. saya perlu tau bagian mana di kalimat user ini yang benar-benar menyatakan kalimat permintaan atau kalimat tanya. respon dengan memberikan langsung kalimat tanya yang sebenarnya dalam bentuk plain teks tanpa karakter bulleting atau newline. "
    "Tolong bedakan penggunaan kata tanya apa, bagaimana, berapa, kapan, siapa, kenapa, mengapa. Terutama bedakan orang yang menanyakan sebuah nilai/entitas dengan menanyakan sebuah proses untuk mendapatkan entitas. "
    "1. Apa: Menanyakan benda, nama data, atau penjelasan umum. contoh: Apa saja variabel yang tersedia dalam survei ini? "
    "2. Bagaimana: Menanyakan cara, proses, atau metodologi. Contoh: Bagaimana cara mengunduh data mentah dari website? "
    "3. Berapa: Menanyakan jumlah, nilai, atau durasi waktu. Contoh: Berapa nilai inflasi tahunan di Provinsi Bali? "
    "4. Kapan: Menanyakan waktu kejadian atau jadwal rilis. Contoh: Kapan data terbaru tahun 2025 akan dirilis? "
    "5. Siapa: Menanyakan subjek (orang/instansi/populasi). Contoh: Siapa responden utama dalam Survei Angkatan Kerja? "
    "6. Mengapa / Kenapa	Menanyakan alasan atau penyebab. Contoh: Mengapa terdapat perbedaan angka antara data pusat dan daerah? "

    "Kadang dalam sapaan atau intro di awal user menyebutkan data/entitas yang diminta, lalu kalimat tanya hanya berupa kata ganti yang menunjuk ke entitas di kalimat awal, maka perlu distruktur ulang menjadi kalimat yanya yang jelas entitas apa yang ingin ditanyakan. "

    "ini kalimat yang perlu dicek: "
    " "
)

# =========================
# Load file XLSX
# =========================
USERDATA = pd.read_excel(INPUT_PATH)

# Pastikan kolom paraphrase ada
if "paraphrase" not in USERDATA.columns:
    USERDATA["paraphrase"] = ""

# =========================
# Buat client Gemini
# =========================
client = genai.Client(api_key=API_KEY)

# =========================
# Looping tiap row
# =========================
for idx, row in USERDATA.iterrows():
    SENTENCE = str(row["question"])  # ambil kolom question
    prompt = PERINTAH + SENTENCE

    try:
        # Request ke Gemini API
        response = client.models.generate_content(
            model="gemini-2.5-flash", #gemini-2.5-flash, gemini-2.5-pro
            contents=prompt
        )

        paraphrase_text = response.text.strip()

        # Masukkan hasil ke kolom paraphrase di baris yang sesuai
        USERDATA.at[idx, "paraphrase"] = paraphrase_text

        # Optional: print progress
        print(f"[{idx+1}/{len(USERDATA)}] Done")

        # Delay sebentar supaya tidak terlalu agresif
        time.sleep(0.1)

    except Exception as e:
        print(f"Error at row {idx}: {e}")
        USERDATA.at[idx, "paraphrase"] = "ERROR"

# =========================
# Simpan kembali ke XLSX
# =========================
USERDATA.to_excel(OUTPUT_PATH, index=False)
print(f"Done! File tersimpan di: {OUTPUT_PATH}")